In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# 1. 加载数据
print("正在加载数据...")
# 修正文件路径
df_original = pd.read_excel('../data\quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)  # 使用相同的缩放器

# 最优参数:
# n_estimators: 500
# max_depth: 19
# min_samples_split: 2
# min_samples_leaf: 1
# max_features: sqrt



# 6. 创建并训练模型
print("\n开始训练随机森林模型...")
model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=500,
    max_depth=19,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
))

# 使用训练数据训练
model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 7. 在测试集上进行预测
y_pred = model.predict(X_test_scaled)

# 8. 评估模型性能
print("\n模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

In [ ]:
# 9. 可视化预测结果与真实值的对比 (使用折线图)
print("\n绘制预测结果与真实值的对比图...")
import matplotlib.pyplot as plt
# 创建一个包含5个子图的大图
fig, axes = plt.subplots(nrows=len(target_vars), figsize=(12, 4*len(target_vars)))
fig.tight_layout(pad=3.0)

# 在原始标度上创建一个数据点索引
x_indices = np.arange(len(y_test))

for i, (var, ax) in enumerate(zip(target_vars, axes)):
    # 绘制真实值 (折线图)
    ax.plot(x_indices, y_test.iloc[:, i].values, 'b-', alpha=0.5, label='真实数据')
    # 绘制预测值 (折线图)
    ax.plot(x_indices, y_pred[:, i], 'r-', alpha=0.5, label='预测数据')
    
    # 设置图形标题和标签
    ax.set_title(f'{var} 预测结果与真实值对比', fontsize=14)
    ax.set_xlabel('样本索引', fontsize=12)
    ax.set_ylabel(f'{var} 值', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # 添加R²指标文本
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes, 
            bbox=dict(facecolor='white', alpha=0.8), fontsize=12)

plt.tight_layout()
plt.show()